In [21]:
import datetime
import os
import subprocess
import MetaTrader5 as mt5
import numpy as np
import pandas as pd
import pytz
from openpyxl.utils import get_column_letter

# ==============================================================================
# CONFIGURATION
# ==============================================================================
SYMBOL = "XAUUSD"                                  # Broker symbol (e.g., "GOLD" or "XAUUSD")
TARGET_TIMEZONE = pytz.timezone("Asia/Singapore")  # UTC+8
EXCEL_FILE = "XAUUSD_Daily_Sessions.xlsx"
HTML_OUTPUT_FILE = "index.html"
RECENT_WINDOW_DAYS = 25                            # Window size for Tables 1, 2, and 5
FORCE_REBUILD_EXCEL = True                         # True to overwrite previous distorted data
AUTO_GIT_PUSH = False                              # Set to True to auto-push to GitHub

# Refined Institutional Session Schedule (UTC+8)
SESSIONS_CONFIG = {
    "Sydney_Open":          (6, 0, 8, 0, False),
    "Tokyo_Open":           (8, 0, 9, 0, False),
    "Shanghai_Open":        (9, 0, 12, 0, False),
    "Asia_Lunch":           (12, 0, 14, 0, False),
    "Pre_London":           (14, 0, 15, 30, False),
    "London_Open":          (15, 30, 18, 0, False),
    "Euro_Afternoon":       (18, 0, 20, 0, False),
    "US_PreMarket_News":    (20, 0, 21, 30, False),
    "NY_Open":              (21, 30, 23, 0, False),
    "London_Fix_Rebalance": (23, 0, 0, 30, True),   # Crosses midnight (23:00 - 00:30)
    "NY_Lunch":             (0, 30, 1, 30, False),
    "NY_Afternoon":         (1, 30, 4, 0, False),
    "Daily_Rollover_Close": (4, 0, 5, 0, False),
}

# ==============================================================================
# 1. MT5 DATA EXTRACTION WITH TIMEZONE AUTO-CALIBRATION
# ==============================================================================

def initialize_mt5():
    if not mt5.initialize():
        print(f"MT5 Initialization error: {mt5.last_error()}")
        return False
    return True


def get_broker_utc_offset(symbol):
    """
    Calculates broker's server UTC offset in seconds.
    """
    tick = mt5.symbol_info_tick(symbol)
    if tick is None:
        return 3 * 3600  # Default to UTC+3 (EEST)
    
    server_time = datetime.datetime.fromtimestamp(tick.time)
    utc_time = datetime.datetime.now(datetime.timezone.utc).replace(tzinfo=None)
    
    # Calculate offset in whole hours
    offset_hours = round((server_time - utc_time).total_seconds() / 3600.0)
    print(f"Detected Broker Server Offset: UTC{'+' if offset_hours >= 0 else ''}{offset_hours}")
    return offset_hours * 3600


def fetch_historical_m5_data(symbol, force_deep=False):
    if not mt5.symbol_select(symbol, True):
        print(f"Failed to select symbol {symbol}")
        return None

    broker_offset_sec = get_broker_utc_offset(symbol)
    bars_to_fetch = 90000 if force_deep else 6000
    
    print(f"Fetching {bars_to_fetch:,} M5 bars for {symbol}...")
    rates = mt5.copy_rates_from_pos(symbol, mt5.TIMEFRAME_M5, 0, bars_to_fetch)
    
    if rates is None or len(rates) == 0:
        print(f"Error fetching data: {mt5.last_error()}")
        return None

    print(f"Retrieved {len(rates):,} raw M5 candles.")
    df = pd.DataFrame(rates)
    
    # Adjust MT5 server timestamps to true UTC
    df["time_utc"] = pd.to_datetime(df["time"] - broker_offset_sec, unit="s", utc=True)
    df["time_local"] = df["time_utc"].dt.tz_convert(TARGET_TIMEZONE)
    
    # 06:00 AM to 05:59 AM next day is one complete trading day
    df["trading_day_dt"] = df["time_local"].apply(lambda t: (t - datetime.timedelta(hours=6)).date())
    df["trading_day"] = df["trading_day_dt"].astype(str)
    
    return df


def assign_session(dt):
    t = dt.time()
    if t >= datetime.time(23, 0) or t < datetime.time(0, 30):
        return "London_Fix_Rebalance"
    
    for name, (sh, sm, eh, em, crosses) in SESSIONS_CONFIG.items():
        if crosses:
            continue
        start = datetime.time(sh, sm)
        end = datetime.time(eh, em)
        if start <= t < end:
            return name
            
    return "Off_Hours"


def process_session_metrics(df):
    df["session"] = df["time_local"].apply(assign_session)
    valid_data = df[df["session"] != "Off_Hours"].copy()

    if valid_data.empty:
        return pd.DataFrame()

    session_summary = (
        valid_data.groupby(["trading_day", "trading_day_dt", "session"], as_index=False)
        .agg(
            open=("open", "first"),
            high=("high", "max"),
            low=("low", "min"),
            close=("close", "last"),
            volume=("tick_volume", "sum"),
            session_start=("time_local", "first"),
            session_end=("time_local", "last")
        )
    )

    session_summary["range_usd"] = (session_summary["high"] - session_summary["low"]).round(2)
    session_summary["net_change"] = (session_summary["close"] - session_summary["open"]).round(2)
    session_summary["bias"] = np.where(
        session_summary["net_change"] > 0, "BULLISH",
        np.where(session_summary["net_change"] < 0, "BEARISH", "NEUTRAL")
    )

    # Weekend filter
    session_summary["weekday"] = session_summary["trading_day_dt"].apply(lambda d: d.weekday())
    session_summary = session_summary[session_summary["weekday"] < 5]

    # Filter out partial days with fewer than 6 sessions
    session_counts = session_summary.groupby("trading_day")["session"].transform("count")
    session_summary = session_summary[session_counts >= 6].copy()

    # Calculate True HOD / LOD Flags
    day_extremes = (
        valid_data.groupby("trading_day")
        .agg(day_high=("high", "max"), day_low=("low", "min"))
        .reset_index()
    )
    
    merged = pd.merge(session_summary, day_extremes, on="trading_day")
    merged["is_HOD"] = merged["high"] == merged["day_high"]
    merged["is_LOD"] = merged["low"] == merged["day_low"]

    merged.drop(columns=["trading_day_dt", "weekday", "day_high", "day_low"], inplace=True)

    session_order = list(SESSIONS_CONFIG.keys())
    merged["session_rank"] = merged["session"].apply(
        lambda x: session_order.index(x) if x in session_order else 99
    )
    merged = merged.sort_values(by=["trading_day", "session_rank"]).drop(columns=["session_rank"])
    
    return merged

# ==============================================================================
# 2. EXCEL PERSISTENCE
# ==============================================================================

def append_and_load_master_excel(new_df, filename, rebuild=False):
    if new_df is not None and not new_df.empty:
        export_df = new_df.copy()
        if pd.api.types.is_datetime64_any_dtype(export_df["session_start"]):
            export_df["session_start"] = export_df["session_start"].dt.strftime("%Y-%m-%d %H:%M")
        if pd.api.types.is_datetime64_any_dtype(export_df["session_end"]):
            export_df["session_end"] = export_df["session_end"].dt.strftime("%Y-%m-%d %H:%M")
    else:
        export_df = pd.DataFrame()

    if os.path.exists(filename) and not rebuild:
        existing_df = pd.read_excel(filename)
        if "trading_day" in existing_df.columns:
            existing_df["_dt"] = pd.to_datetime(existing_df["trading_day"])
            existing_df = existing_df[existing_df["_dt"].dt.weekday < 5].drop(columns=["_dt"])
        
        combined_df = pd.concat([existing_df, export_df], ignore_index=True)
        combined_df = combined_df.drop_duplicates(
            subset=["trading_day", "session"], keep="last"
        )
    else:
        combined_df = export_df

    combined_df = combined_df.sort_values(by=["trading_day", "session_start"])
    combined_df["is_HOD"] = combined_df["is_HOD"].astype(bool)
    combined_df["is_LOD"] = combined_df["is_LOD"].astype(bool)

    with pd.ExcelWriter(filename, engine="openpyxl") as writer:
        combined_df.to_excel(writer, sheet_name="Session_Logs", index=False)
        worksheet = writer.sheets["Session_Logs"]

        for col_idx, col in enumerate(worksheet.columns, start=1):
            max_len = max(len(str(cell.value or "")) for cell in col)
            col_letter = get_column_letter(col_idx)
            worksheet.column_dimensions[col_letter].width = max(max_len + 4, 12)

    total_days = combined_df['trading_day'].nunique()
    print(f"Master Excel updated: '{filename}' ({len(combined_df):,} records across {total_days} full trading days).")
    return combined_df

# ==============================================================================
# 3. HTML DASHBOARD GENERATOR (5 TABLES)
# ==============================================================================

HTML_TEMPLATE = """<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="UTF-8">
  <meta name="viewport" content="width=device-width, initial-scale=1.0">
  <title>XAUUSD Institutional Session Dashboard</title>
  <style>
    :root {{
      --bg-header: #131c2e;
      --border-color: #e2e8f0;
      --header-divider: #23334d;
      --text-main: #0f172a;
      --text-muted: #94a3b8;
    }}
    * {{
      box-sizing: border-box;
      margin: 0;
      padding: 0;
      font-family: -apple-system, BlinkMacSystemFont, "Segoe UI", Roboto, Helvetica, Arial, sans-serif;
    }}
    body {{
      background-color: #f8fafc;
      padding: 24px;
      color: var(--text-main);
    }}
    .dashboard-grid {{
      display: grid;
      grid-template-columns: 1fr 1fr;
      gap: 20px;
      margin-bottom: 24px;
    }}
    @media (max-width: 1200px) {{
      .dashboard-grid {{
        grid-template-columns: 1fr;
      }}
    }}
    .table-container {{
      max-width: 100%;
      overflow-x: auto;
      background: #ffffff;
      border-radius: 8px;
      box-shadow: 0 4px 12px rgba(0, 0, 0, 0.08);
      border: 1px solid var(--border-color);
      margin-bottom: 24px;
    }}
    table {{
      width: 100%;
      table-layout: fixed;
      border-collapse: collapse;
      text-align: center;
      font-size: 13px;
      white-space: nowrap;
    }}
    thead tr.header-title th {{
      background-color: var(--bg-header);
      color: #ffffff;
      text-align: left;
      padding: 14px 18px;
      font-size: 13.5px;
      font-weight: 600;
      letter-spacing: 0.3px;
    }}
    thead tr.header-slots th {{
      background-color: var(--bg-header);
      color: #f1f5f9;
      padding: 10px 8px;
      font-size: 11.5px;
      font-weight: 700;
      border-left: 1px solid var(--header-divider);
    }}
    thead th.no-border-left {{
      border-left: none !important;
    }}
    tbody tr {{
      border-bottom: 1px solid var(--border-color);
      height: 44px;
      transition: background-color 0.15s ease;
    }}
    tbody tr:hover {{
      background-color: #f8fafc;
    }}
    tbody td {{
      padding: 8px 6px;
      vertical-align: middle;
      font-size: 13px;
      border-left: 1px solid #f1f5f9;
    }}
    tbody td.session-name-cell, tbody td.date-cell {{
      font-weight: 600;
      color: #334155;
      padding-left: 16px;
      padding-right: 16px;
      text-align: left;
      border-left: none;
      font-size: 13px;
    }}
    tbody td.metric-cell {{
      font-weight: 700;
      color: #1e293b;
      font-size: 13px;
    }}
    .badge {{
      display: inline-block;
      padding: 4px 6px;
      border-radius: 6px;
      font-weight: 700;
      font-size: 11px;
      letter-spacing: 0.4px;
      min-width: 64px;
    }}
    .badge-bullish {{ background-color: #d1fae5; color: #065f46; }}
    .badge-bearish {{ background-color: #ffe4e6; color: #9f1239; }}
    .badge-neutral {{ background-color: #f1f5f9; color: #475569; }}
    .progress-bar-container {{
      background-color: #e2e8f0;
      border-radius: 4px;
      width: 100%;
      height: 5px;
      margin-top: 4px;
      overflow: hidden;
    }}
    .progress-fill-bull {{ background-color: #10b981; height: 100%; }}
    .progress-fill-bear {{ background-color: #f43f5e; height: 100%; }}
  </style>
</head>
<body>

  <!-- ROW 1: 25-DAY METRICS -->
  <div class="dashboard-grid">
    
    <!-- Table 1: 25-Day Directional Rates -->
    <div class="table-container" style="margin-bottom: 0;">
      <table>
        <thead>
          <tr class="header-title">
            <th colspan="5">1. Session Bias & Directional Rate (Past {recent_count} Trading Days)</th>
          </tr>
          <tr class="header-slots">
            <th class="no-border-left" style="width: 26%; text-align: left; padding-left: 16px;">SESSION</th>
            <th style="width: 20%;">BULLISH %</th>
            <th style="width: 20%;">BEARISH %</th>
            <th style="width: 17%;">NEUTRAL %</th>
            <th style="width: 17%;">AVG RANGE</th>
          </tr>
        </thead>
        <tbody>
          {recent_bias_rows}
        </tbody>
      </table>
    </div>

    <!-- Table 2: 25-Day HOD / LOD Profile -->
    <div class="table-container" style="margin-bottom: 0;">
      <table>
        <thead>
          <tr class="header-title">
            <th colspan="5">2. Session HOD / LOD Probability Profile (Past {recent_count} Trading Days)</th>
          </tr>
          <tr class="header-slots">
            <th class="no-border-left" style="width: 26%; text-align: left; padding-left: 16px;">SESSION</th>
            <th style="width: 18%;">HOD COUNT</th>
            <th style="width: 19%;">HOD PROB %</th>
            <th style="width: 18%;">LOD COUNT</th>
            <th style="width: 19%;">LOD PROB %</th>
          </tr>
        </thead>
        <tbody>
          {recent_hod_lod_rows}
        </tbody>
      </table>
    </div>

  </div>

  <!-- ROW 2: LIFETIME HISTORICAL METRICS -->
  <div class="dashboard-grid">
    
    <!-- Table 3: Lifetime Directional Rates -->
    <div class="table-container" style="margin-bottom: 0;">
      <table>
        <thead>
          <tr class="header-title">
            <th colspan="5">3. Lifetime Session Bias & Directional Rate (Total {total_lifetime_days} Trading Days)</th>
          </tr>
          <tr class="header-slots">
            <th class="no-border-left" style="width: 26%; text-align: left; padding-left: 16px;">SESSION</th>
            <th style="width: 20%;">BULLISH %</th>
            <th style="width: 20%;">BEARISH %</th>
            <th style="width: 17%;">NEUTRAL %</th>
            <th style="width: 17%;">AVG RANGE</th>
          </tr>
        </thead>
        <tbody>
          {lifetime_bias_rows}
        </tbody>
      </table>
    </div>

    <!-- Table 4: Lifetime HOD / LOD Profile -->
    <div class="table-container" style="margin-bottom: 0;">
      <table>
        <thead>
          <tr class="header-title">
            <th colspan="5">4. Lifetime Session HOD / LOD Profile (Total {total_lifetime_days} Trading Days)</th>
          </tr>
          <tr class="header-slots">
            <th class="no-border-left" style="width: 26%; text-align: left; padding-left: 16px;">SESSION</th>
            <th style="width: 18%;">HOD COUNT</th>
            <th style="width: 19%;">HOD PROB %</th>
            <th style="width: 18%;">LOD COUNT</th>
            <th style="width: 19%;">LOD PROB %</th>
          </tr>
        </thead>
        <tbody>
          {lifetime_hod_lod_rows}
        </tbody>
      </table>
    </div>

  </div>

  <!-- ROW 3: DAILY DETAIL MATRIX -->
  <div class="table-container">
    <table style="table-layout: auto;">
      <thead>
        <tr class="header-title">
          <th colspan="{matrix_total_cols}">5. Daily Session Directional Matrix (Last {recent_count} Days)</th>
        </tr>
        <tr class="header-slots">
          <th class="no-border-left">DATE</th>
          {matrix_headers}
          <th>BULLISH<br>RATE</th>
          <th>BEARISH<br>RATE</th>
        </tr>
      </thead>
      <tbody>
        {matrix_rows}
      </tbody>
    </table>
  </div>

</body>
</html>
"""

def build_bias_rows(df, sessions):
    rows = []
    for s in sessions:
        s_data = df[df["session"] == s]
        count = len(s_data)
        
        if count > 0:
            bull_c = (s_data["bias"] == "BULLISH").sum()
            bear_c = (s_data["bias"] == "BEARISH").sum()
            neut_c = (s_data["bias"] == "NEUTRAL").sum()
            avg_range = s_data["range_usd"].mean()

            bull_pct = (bull_c / count) * 100
            bear_pct = (bear_c / count) * 100
            neut_pct = (neut_c / count) * 100
        else:
            bull_c = bear_c = neut_c = 0
            bull_pct = bear_pct = neut_pct = 0.0
            avg_range = 0.0

        rows.append(f"""
          <tr>
            <td class="session-name-cell">{s.replace('_', ' ')}</td>
            <td class="metric-cell" style="color: #065f46;">
              {bull_pct:.1f}% ({bull_c})
              <div class="progress-bar-container"><div class="progress-fill-bull" style="width: {bull_pct}%;"></div></div>
            </td>
            <td class="metric-cell" style="color: #9f1239;">
              {bear_pct:.1f}% ({bear_c})
              <div class="progress-bar-container"><div class="progress-fill-bear" style="width: {bear_pct}%;"></div></div>
            </td>
            <td>{neut_pct:.1f}% ({neut_c})</td>
            <td class="metric-cell">${avg_range:.2f}</td>
          </tr>
        """)
    return "\n".join(rows)


def build_hod_lod_rows(df, sessions, num_days):
    rows = []
    for s in sessions:
        s_data = df[df["session"] == s]
        hod_c = s_data["is_HOD"].sum() if not s_data.empty else 0
        lod_c = s_data["is_LOD"].sum() if not s_data.empty else 0

        hod_pct = (hod_c / num_days * 100) if num_days > 0 else 0.0
        lod_pct = (lod_c / num_days * 100) if num_days > 0 else 0.0

        rows.append(f"""
          <tr>
            <td class="session-name-cell">{s.replace('_', ' ')}</td>
            <td class="metric-cell">{hod_c}</td>
            <td class="metric-cell" style="color: #047857;">{hod_pct:.1f}%</td>
            <td class="metric-cell">{lod_c}</td>
            <td class="metric-cell" style="color: #b91c1c;">{lod_pct:.1f}%</td>
          </tr>
        """)
    return "\n".join(rows)


def generate_dashboard(master_df, output_file=HTML_OUTPUT_FILE):
    if master_df.empty:
        return

    sessions = list(SESSIONS_CONFIG.keys())
    all_days = sorted(master_df["trading_day"].unique(), reverse=True)
    total_lifetime_days = len(all_days)

    recent_days_list = all_days[:RECENT_WINDOW_DAYS]
    recent_df = master_df[master_df["trading_day"].isin(recent_days_list)]
    recent_count = len(recent_days_list)

    recent_bias_rows = build_bias_rows(recent_df, sessions)
    recent_hod_lod_rows = build_hod_lod_rows(recent_df, sessions, recent_count)
    lifetime_bias_rows = build_bias_rows(master_df, sessions)
    lifetime_hod_lod_rows = build_hod_lod_rows(master_df, sessions, total_lifetime_days)

    matrix_headers = "".join([f'<th>{s.replace("_", " ")}</th>' for s in sessions])
    matrix_total_cols = 1 + len(sessions) + 2
    matrix_rows = []

    for day in recent_days_list:
        day_data = recent_df[recent_df["trading_day"] == day]
        bull_count = 0
        bear_count = 0
        cells = [f'<td class="date-cell">{day}</td>']

        for s in sessions:
            row_match = day_data[day_data["session"] == s]
            bias = row_match["bias"].values[0] if not row_match.empty else "NEUTRAL"

            if bias == "BULLISH":
                bull_count += 1
                badge_class = "badge-bullish"
            elif bias == "BEARISH":
                bear_count += 1
                badge_class = "badge-bearish"
            else:
                badge_class = "badge-neutral"

            badge_html = f'<span class="badge {badge_class}">{bias}</span>'
            cells.append(f'<td>{badge_html}</td>')

        total_sessions = len(sessions)
        bull_rate = f"{(bull_count / total_sessions * 100):.1f}%"
        bear_rate = f"{(bear_count / total_sessions * 100):.1f}%"

        cells.append(f'<td class="metric-cell">{bull_rate}</td>')
        cells.append(f'<td class="metric-cell">{bear_rate}</td>')

        matrix_rows.append(f'<tr>{"".join(cells)}</tr>')

    full_html = HTML_TEMPLATE.format(
        recent_count=recent_count,
        total_lifetime_days=total_lifetime_days,
        recent_bias_rows=recent_bias_rows,
        recent_hod_lod_rows=recent_hod_lod_rows,
        lifetime_bias_rows=lifetime_bias_rows,
        lifetime_hod_lod_rows=lifetime_hod_lod_rows,
        matrix_total_cols=matrix_total_cols,
        matrix_headers=matrix_headers,
        matrix_rows="\n".join(matrix_rows)
    )

    with open(output_file, "w", encoding="utf-8") as f:
        f.write(full_html)
    print(f"Generated dashboard: 25-Day view vs {total_lifetime_days}-Day Lifetime view.")

# ==============================================================================
# 4. GITHUB DEPLOYMENT
# ==============================================================================

def push_to_github():
    try:
        subprocess.run(["git", "add", EXCEL_FILE, HTML_OUTPUT_FILE], check=True)
        subprocess.run(["git", "commit", "-m", f"Auto-update: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}"], check=True)
        subprocess.run(["git", "push"], check=True)
        print("Successfully pushed changes to GitHub.")
    except Exception as e:
        print(f"Git push skipped or failed: {e}")

# ==============================================================================
# MAIN PIPELINE
# ==============================================================================

if __name__ == "__main__":
    new_data = None
    if initialize_mt5():
        df_raw = fetch_historical_m5_data(SYMBOL, force_deep=FORCE_REBUILD_EXCEL)
        mt5.shutdown()

        if df_raw is not None:
            new_data = process_session_metrics(df_raw)

    master_dataset = append_and_load_master_excel(new_data, EXCEL_FILE, rebuild=FORCE_REBUILD_EXCEL)
    generate_dashboard(master_dataset, HTML_OUTPUT_FILE)
    
    if AUTO_GIT_PUSH:
        push_to_github()
        
    print("Pipeline completed successfully.")

Detected Broker Server Offset: UTC-10
Fetching 90,000 M5 bars for XAUUSD...
Retrieved 90,000 raw M5 candles.
Master Excel updated: 'XAUUSD_Daily_Sessions.xlsx' (3,850 records across 328 full trading days).
Generated dashboard: 25-Day view vs 328-Day Lifetime view.
Pipeline completed successfully.
